# 01 - Ingest: Databricks Docs Corpus
## Databricks Expert Agent Project

**What this notebook does:**
1. Sets up Unity Catalog schema and volume
2. Pulls every Databricks doc page URL from the official sitemap
3. Scrapes them in parallel using Spark workers (distributed - this is the Databricks way)
4. Writes the raw text to a managed Delta table in Unity Catalog

**Output:** `chatbot.rag_chatbot.raw_docs` - the foundation the agent learns from

In [0]:
# -- Project config ----------------------------------------------
CATALOG     = "chatbot"
SCHEMA      = "rag_chatbot"
VOLUME      = "raw_docs"

# Derived - don't change these
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
RAW_TABLE   = f"{CATALOG}.{SCHEMA}.raw_docs"

print(f"Catalog : {CATALOG}")
print(f"Schema  : {CATALOG}.{SCHEMA}")
print(f"Volume  : {VOLUME_PATH}")
print(f"Table   : {RAW_TABLE}")

## Cluster Dependencies
The following libraries must be installed at the **cluster level** (not notebook level):
- `requests` - HTTP calls to fetch doc pages
- `beautifulsoup4` - HTML parsing
- `lxml` - parsing engine for BeautifulSoup

**To install:** Compute -> your cluster -> Libraries tab -> Install New -> PyPI

###Creating Catalog, Schema, and Volume

In [0]:
spark.sql(f'create catalog if not exists {CATALOG}')
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
print(f'OK Catalog: {CATALOG}')
print(f"OK Schema : {CATALOG}.{SCHEMA}")
print(f"OK Volume : {VOLUME_PATH}")

In [0]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import os
import re

In [0]:
# -- Azure Databricks TOC crawl ----------------------------------
TOC_URL  = "https://learn.microsoft.com/en-us/azure/databricks/toc.json"
BASE_URL = "https://learn.microsoft.com/en-us/azure/databricks/"

def normalize_href(href: str):
    if not href:
        return None

    href = href.split("?")[0].split("#")[0].strip()
    if not href:
        return None

    if href.startswith("https://") or href.startswith("http://"):
        if "learn.microsoft.com" in href and "/azure/databricks" in href:
            return href.rstrip("/")
        return None

    if href.startswith("/en-us/azure/databricks"):
        return "https://learn.microsoft.com" + href.rstrip("/")

    if href.startswith("/azure/databricks"):
        return "https://learn.microsoft.com/en-us" + href.rstrip("/")

    if not href.startswith("/"):
        return BASE_URL.rstrip("/") + "/" + href.lstrip("/").rstrip("/")

    return None

def extract_urls(node, urls=None):
    if urls is None:
        urls = []

    if isinstance(node, dict):
        href = node.get("href")
        normalized = normalize_href(href) if href else None

        if normalized and not node.get("redirect"):
            urls.append(normalized)

        for key in ("items", "children"):
            for child in node.get(key, []):
                extract_urls(child, urls)

    elif isinstance(node, list):
        for item in node:
            extract_urls(item, urls)

    return urls

print(f"Fetching TOC from {TOC_URL} ...")
resp = requests.get(
    TOC_URL,
    timeout=30,
    headers={"User-Agent": "DatabricksAgentProject/1.0"}
)
resp.raise_for_status()

toc = resp.json()
raw_urls = extract_urls(toc)

seen = set()
ms_urls = []
for u in raw_urls:
    if u not in seen:
        seen.add(u)
        ms_urls.append(u)

all_doc_urls = ms_urls

print(f"Azure Databricks MS Learn docs: {len(ms_urls):,}")
print("Ready to scrape.")
print(ms_urls[:10])

In [0]:
# -- Page scraper function ---------------------------------------
def scrape_doc_page(url: str):
    try:
        r = requests.get(
            url,
            timeout=30,
            headers={"User-Agent": "DatabricksAgentProject/1.0"}
        )
        r.raise_for_status()

        soup = BeautifulSoup(r.text, "lxml")

        for tag in soup(["script", "style", "noscript", "svg", "img"]):
            tag.decompose()

        title = soup.title.get_text(" ", strip=True) if soup.title else url

        main = (
            soup.find("main")
            or soup.find("article")
            or soup.find("div", {"role": "main"})
            or soup.body
        )

        content = main.get_text(" ", strip=True) if main else ""
        content = " ".join(content.split())

        return (
            url,
            title,
            content,
            "ok",
            datetime.utcnow().isoformat()
        )

    except Exception as e:
        return (
            url,
            None,
            None,
            f"error: {str(e)[:500]}",
            datetime.utcnow().isoformat()
        )

In [0]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StructType, StructField, StringType

# Define the return schema for our scraper UDF
result_schema = StructType([
    StructField("url",          StringType(), True),
    StructField("title",        StringType(), True),
    StructField("content",      StringType(), True),
    StructField("status",       StringType(), True),
    StructField("scraped_date", StringType(), True),
])

# Wrap scraper function as a Spark UDF
scrape_udf = udf(scrape_doc_page, result_schema)

# Build URL DataFrame
urls_df = spark.createDataFrame([(u,) for u in all_doc_urls], ["url"])

print(f"Scraping {urls_df.count():,} URLs across workers...")
print("Expect several minutes depending on cluster size.\n")

In [0]:
from datetime import datetime# Apply the UDF across Spark workers

scraped_df = urls_df.select(scrape_udf(col("url")).alias("result")).select("result.*")

# Split successful vs failed
ok_df  = scraped_df.filter(col("status") == "ok").cache()
bad_df = scraped_df.filter(col("status") != "ok").cache()

print(f"OK Successful : {ok_df.count():,}")
print(f"WARN Failed     : {bad_df.count():,}")

In [0]:
bad_df.select("status").groupBy("status").count().display()

In [0]:
# -- Internal ADO Wiki / TSG Markdown ingestion ------------------------------
# Upload your cloned wiki repo folder into this Volume path first.
# Example final path:
# /Volumes/chatbot/rag_chatbot/raw_docs/internal_wiki/AzureDataBricks.wiki

WIKI_ROOT = f"{VOLUME_PATH}/internal_wiki/AzureDataBricks.wiki"

def clean_markdown(text: str) -> str:
    # Keep the useful content, remove markdown noise
    text = re.sub(r"```.*?```", " ", text, flags=re.DOTALL)      # code blocks
    text = re.sub(r"`([^`]*)`", r"\1", text)                     # inline code markers
    text = re.sub(r"!\[.*?\]\(.*?\)", " ", text)                 # images
    text = re.sub(r"\[(.*?)\]\((.*?)\)", r"\1", text)            # links keep text
    text = re.sub(r"#+\s*", "", text)                            # headings
    text = re.sub(r"[*_~]", "", text)                            # markdown emphasis
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

wiki_rows = []

if os.path.exists(WIKI_ROOT):
    for root, dirs, files in os.walk(WIKI_ROOT):
        for file in files:
            if not file.lower().endswith((".md", ".markdown")):
                continue

            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, WIKI_ROOT)

            try:
                with open(full_path, "r", encoding="utf-8", errors="ignore") as f:
                    raw_md = f.read()

                content = clean_markdown(raw_md)

                if len(content) < 100:
                    continue

                title = rel_path.replace("\\", "/").replace(".md", "")
                url = f"internal_wiki://AzureDataBricks.wiki/{title}"

                wiki_rows.append((url, title, content, "ok", datetime.utcnow().isoformat()))

            except Exception as e:
                wiki_rows.append((
                    f"internal_wiki://AzureDataBricks.wiki/{rel_path}",
                    rel_path,
                    None,
                    f"error: {str(e)[:500]}",
                    datetime.utcnow().isoformat()
                ))

    print(f"Internal wiki markdown docs found: {len(wiki_rows):,}")
else:
    print(f"Wiki path not found: {WIKI_ROOT}")
    print("Skipping internal wiki ingestion.")

wiki_schema = result_schema

if wiki_rows:
    wiki_df = spark.createDataFrame(wiki_rows, schema=wiki_schema)
    wiki_ok_df = wiki_df.filter(col("status") == "ok")
    wiki_bad_df = wiki_df.filter(col("status") != "ok")

    print(f"OK Wiki successful : {wiki_ok_df.count():,}")
    print(f"WARN Wiki failed     : {wiki_bad_df.count():,}")

    if wiki_bad_df.count() > 0:
        display(wiki_bad_df)
else:
    wiki_ok_df = spark.createDataFrame([], schema=wiki_schema)


In [0]:
# -- Combine Microsoft Learn docs + internal wiki docs -----------------------
combined_df = ok_df.unionByName(wiki_ok_df)

print(f"Microsoft Learn docs : {ok_df.count():,}")
print(f"Internal wiki docs   : {wiki_ok_df.count():,}")
print(f"Combined docs        : {combined_df.count():,}")

(
    combined_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RAW_TABLE)
)

print(f"OK Written to : {RAW_TABLE}")
print(f"OK Row count  : {spark.table(RAW_TABLE).count():,}")

In [0]:
from pyspark.sql.functions import length

df = spark.table(RAW_TABLE)
df.printSchema()

print(f"\nTotal rows: {df.count():,}")

print("\nContent length distribution:")
display(
    df.select(
        "title",
        length(col("content")).alias("content_chars")
    )
    .orderBy(col("content_chars").desc())
)

In [0]:
%skip
display(spark.sql("""
SELECT
  CASE
    WHEN url LIKE 'internal_wiki://%' THEN 'internal_wiki'
    ELSE 'microsoft_learn'
  END AS source_guess,
  COUNT(*) AS docs
FROM chatbot.rag_chatbot.raw_docs
GROUP BY source_guess
"""))

In [0]:
%skip
display(spark.sql("""
SELECT source, source_type, COUNT(*) AS chunks
FROM chatbot.rag_chatbot.doc_chunks
GROUP BY source, source_type
ORDER BY chunks DESC
"""))